In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

# read the save sate from 01_Label_merchants.ipynb
seed_set = pd.read_parquet("../../data/curated/seed_set_merchants.parquet")
unlabeled = pd.read_parquet("../../data/curated/unlabeled_merchants.parquet")

num_cols = ["take_rate", "avg_order_value", "log_total_revenue", "log_n_transactions"]

# re-create the pipeline for preprocessing and modeling
preprocess = ColumnTransformer([("num", StandardScaler(), num_cols)])
model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000))
])

print(f"seed_set: {len(seed_set)} rows, unlabeled: {len(unlabeled)} rows")

seed_set: 48 rows, unlabeled: 3978 rows


# NAIVE ITERATIVE SELF-TRAINING

In [2]:
n_rounds = 10          # max number of rounds
high_conf = 0.9        # confidence threshold to label as fraud
low_conf = 0.1          # confidence threshold to label as clean

# fixed train/test split before starting the loop
tr, te = train_test_split(
    seed_set, test_size=0.3, random_state=42, stratify=seed_set["is_fraud"]
)

labeled_pool = tr.copy()              # training set that will grow each round
unlabeled_pool = unlabeled.copy()     # unlabeled pool that will shrink each round

round_results = []  # track AUC + counts per round

# train model on the current labeled pool
for round_i in range(n_rounds):
    m = clone(model)
    m.fit(labeled_pool[num_cols], labeled_pool["is_fraud"])

    # evaluate on the fixed, untouched test set
    auc = roc_auc_score(te["is_fraud"], m.predict_proba(te[num_cols])[:, 1])

    
    if len(unlabeled_pool) == 0:
        print(f"Round {round_i}: unlabeled pool is empty, stopping.")
        break

    # predict on the remaining unlabeled pool
    probs = m.predict_proba(unlabeled_pool[num_cols])[:, 1]
    unlabeled_pool = unlabeled_pool.copy()
    unlabeled_pool["pred_proba"] = probs

    # filter the confident samples based on the thresholds
    confident_fraud = unlabeled_pool[unlabeled_pool["pred_proba"] > high_conf].assign(is_fraud=1.0)
    confident_clean = unlabeled_pool[unlabeled_pool["pred_proba"] < low_conf].assign(is_fraud=0.0)
    new_confident = pd.concat([confident_fraud, confident_clean]).drop(columns=["pred_proba"])

    round_results.append({
        "round": round_i,
        "auc": auc,
        "labeled_size": len(labeled_pool),
        "new_confident": len(new_confident),
        "remaining_unlabeled": len(unlabeled_pool) - len(new_confident),
    })

    print(f"Round {round_i}: AUC={auc:.3f} | labeled={len(labeled_pool)} "
          f"| +{len(new_confident)} newly confident | remaining unlabeled={len(unlabeled_pool) - len(new_confident)}")

    # if no more confident samples, stop the loop
    if len(new_confident) == 0:
        print(f"Round {round_i}: no samples met the confidence threshold, stopping.")
        break

    # move confident samples from unlabeled to labeled
    labeled_pool = pd.concat([labeled_pool, new_confident], ignore_index=True)
    unlabeled_pool = unlabeled_pool.drop(new_confident.index).drop(columns=["pred_proba"], errors="ignore")

# summary
round_results_df = pd.DataFrame(round_results)
print("\n=== Round-by-round summary ===")
print(round_results_df)

print(f"\nAUC at round 0 (original seed_set only): {round_results_df['auc'].iloc[0]:.3f}")
print(f"AUC at final round (after iterative self-training): {round_results_df['auc'].iloc[-1]:.3f}")
print(f"Total samples added to labeled pool: {len(labeled_pool) - len(tr)}")

Round 0: AUC=0.920 | labeled=33 | +1792 newly confident | remaining unlabeled=2186
Round 1: AUC=0.900 | labeled=1825 | +1279 newly confident | remaining unlabeled=907
Round 2: AUC=0.880 | labeled=3104 | +524 newly confident | remaining unlabeled=383
Round 3: AUC=0.820 | labeled=3628 | +228 newly confident | remaining unlabeled=155
Round 4: AUC=0.820 | labeled=3856 | +68 newly confident | remaining unlabeled=87
Round 5: AUC=0.820 | labeled=3924 | +21 newly confident | remaining unlabeled=66
Round 6: AUC=0.820 | labeled=3945 | +4 newly confident | remaining unlabeled=62
Round 7: AUC=0.820 | labeled=3949 | +1 newly confident | remaining unlabeled=61
Round 8: AUC=0.820 | labeled=3950 | +1 newly confident | remaining unlabeled=60
Round 9: AUC=0.820 | labeled=3951 | +0 newly confident | remaining unlabeled=60
Round 9: no samples met the confidence threshold, stopping.

=== Round-by-round summary ===
   round   auc  labeled_size  new_confident  remaining_unlabeled
0      0  0.92            33

## BOOTSTRP ENSEMBLE SELF TRAINING (SINGLE RUN)

In [3]:
from sklearn.utils import resample

n_rounds = 10
n_bootstrap = 5        # number of models per round
high_conf = 0.9        
low_conf = 0.1          
max_std = 0.1            # how much std is allowed for the models to "agree" on a sample
max_add_per_round = 30   # restrict how many samples can be added 

tr, te = train_test_split(
    seed_set, test_size=0.3, random_state=42, stratify=seed_set["is_fraud"]
)

labeled_pool = tr.copy()
unlabeled_pool = unlabeled.copy()

round_results = []
best_auc = -1
best_labeled_pool = None
best_round = -1

# train ensemble of 5 bootstrappd models 
for round_i in range(n_rounds):
    ensemble_models = []
    for b in range(n_bootstrap):
        boot_sample = resample(labeled_pool, random_state=round_i * 100 + b)
        m_b = clone(model)
        m_b.fit(boot_sample[num_cols], boot_sample["is_fraud"])
        ensemble_models.append(m_b)

    # evaluate ensemble on average prediction on the fixed test set
    test_probs = np.array([m_b.predict_proba(te[num_cols])[:, 1] for m_b in ensemble_models])
    test_mean_prob = test_probs.mean(axis=0)
    auc = roc_auc_score(te["is_fraud"], test_mean_prob)

    # track best round
    if auc > best_auc:
        best_auc = auc
        best_labeled_pool = labeled_pool.copy()
        best_round = round_i

    if len(unlabeled_pool) == 0:
        print(f"Round {round_i}: unlabeled pool is empty, stopping.")
        break

    # predict on unlabeled pool with the whole ensemble
    unlabeled_probs = np.array([m_b.predict_proba(unlabeled_pool[num_cols])[:, 1] for m_b in ensemble_models])
    mean_prob = unlabeled_probs.mean(axis=0)
    std_prob = unlabeled_probs.std(axis=0)

    unlabeled_pool = unlabeled_pool.copy()
    unlabeled_pool["mean_prob"] = mean_prob
    unlabeled_pool["std_prob"] = std_prob

    # only keep samples where the models "agree" (std <= max_std) and are confident (mean_prob > high_conf or < low_conf)
    agree_mask = unlabeled_pool["std_prob"] <= max_std
    confident_fraud = unlabeled_pool[agree_mask & (unlabeled_pool["mean_prob"] > high_conf)] \
        .assign(is_fraud=1.0)
    confident_clean = unlabeled_pool[agree_mask & (unlabeled_pool["mean_prob"] < low_conf)] \
        .assign(is_fraud=0.0)

    new_confident = pd.concat([confident_fraud, confident_clean])

    # cap how many are added this round, keep the most confident ones (by distance from 0.5)
    if len(new_confident) > max_add_per_round:
        new_confident = new_confident.assign(
            _confidence=np.abs(new_confident["mean_prob"] - 0.5)
        ).nlargest(max_add_per_round, "_confidence").drop(columns=["_confidence"])

    new_confident = new_confident.drop(columns=["mean_prob", "std_prob"])

    round_results.append({
        "round": round_i,
        "auc": auc,
        "labeled_size": len(labeled_pool),
        "new_confident": len(new_confident),
        "remaining_unlabeled": len(unlabeled_pool) - len(new_confident),
    })

    print(f"Round {round_i}: AUC={auc:.3f} | labeled={len(labeled_pool)} "
          f"| +{len(new_confident)} agreed & confident | remaining unlabeled={len(unlabeled_pool) - len(new_confident)}")

    # stop early if AUC drops meaningfully from the best
    if auc < best_auc - 0.01:
        print(f"Round {round_i}: AUC dropped from best ({best_auc:.3f} at round {best_round}), stopping early.")
        break

    if len(new_confident) == 0:
        print(f"Round {round_i}: no samples met the agreement + confidence threshold, stopping.")
        break

    # move confident, agreed-upon samples into labeled pool
    labeled_pool = pd.concat([labeled_pool, new_confident], ignore_index=True)
    unlabeled_pool = unlabeled_pool.drop(new_confident.index).drop(columns=["mean_prob", "std_prob"], errors="ignore")

# summary
round_results_df = pd.DataFrame(round_results)
print("\n=== Round-by-round summary ===")
print(round_results_df)

print(f"\nBest AUC: {best_auc:.3f} at round {best_round}")
print(f"Labeled pool size at best round: {len(best_labeled_pool)}")
print(f"AUC at round 0 (original seed_set only): {round_results_df['auc'].iloc[0]:.3f}")
print(f"AUC at final round: {round_results_df['auc'].iloc[-1]:.3f}")

Round 0: AUC=0.920 | labeled=33 | +30 agreed & confident | remaining unlabeled=3948
Round 1: AUC=0.920 | labeled=63 | +30 agreed & confident | remaining unlabeled=3918
Round 2: AUC=0.960 | labeled=93 | +30 agreed & confident | remaining unlabeled=3888
Round 3: AUC=0.800 | labeled=123 | +30 agreed & confident | remaining unlabeled=3858
Round 3: AUC dropped from best (0.960 at round 2), stopping early.

=== Round-by-round summary ===
   round   auc  labeled_size  new_confident  remaining_unlabeled
0      0  0.92            33             30                 3948
1      1  0.92            63             30                 3918
2      2  0.96            93             30                 3888
3      3  0.80           123             30                 3858

Best AUC: 0.960 at round 2
Labeled pool size at best round: 93
AUC at round 0 (original seed_set only): 0.920
AUC at final round: 0.800


## STABILITY CHECK (REPEAT 15 TIMES)

In [4]:
def run_iterative_ensemble(seed_set, unlabeled_full, num_cols, model,
                            split_seed, n_rounds=10, n_bootstrap=5,
                            high_conf=0.9, low_conf=0.1, max_std=0.1,
                            max_add_per_round=30):

    tr, te = train_test_split(
        seed_set, test_size=0.3, random_state=split_seed, stratify=seed_set["is_fraud"]
    )

    labeled_pool = tr.copy()
    unlabeled_pool = unlabeled_full.copy()

    best_auc = -1
    best_round = -1
    baseline_auc = None

    for round_i in range(n_rounds):
        ensemble_models = []
        for b in range(n_bootstrap):
            boot_sample = resample(labeled_pool, random_state=split_seed * 1000 + round_i * 100 + b)
            m_b = clone(model)
            m_b.fit(boot_sample[num_cols], boot_sample["is_fraud"])
            ensemble_models.append(m_b)

        test_probs = np.array([m_b.predict_proba(te[num_cols])[:, 1] for m_b in ensemble_models])
        test_mean_prob = test_probs.mean(axis=0)
        auc = roc_auc_score(te["is_fraud"], test_mean_prob)

        if round_i == 0:
            baseline_auc = auc

        if auc > best_auc:
            best_auc = auc
            best_round = round_i

        if len(unlabeled_pool) == 0:
            break

        unlabeled_probs = np.array([m_b.predict_proba(unlabeled_pool[num_cols])[:, 1] for m_b in ensemble_models])
        mean_prob = unlabeled_probs.mean(axis=0)
        std_prob = unlabeled_probs.std(axis=0)

        unlabeled_pool = unlabeled_pool.copy()
        unlabeled_pool["mean_prob"] = mean_prob
        unlabeled_pool["std_prob"] = std_prob

        agree_mask = unlabeled_pool["std_prob"] <= max_std
        confident_fraud = unlabeled_pool[agree_mask & (unlabeled_pool["mean_prob"] > high_conf)].assign(is_fraud=1.0)
        confident_clean = unlabeled_pool[agree_mask & (unlabeled_pool["mean_prob"] < low_conf)].assign(is_fraud=0.0)
        new_confident = pd.concat([confident_fraud, confident_clean])

        if len(new_confident) > max_add_per_round:
            new_confident = new_confident.assign(
                _confidence=np.abs(new_confident["mean_prob"] - 0.5)
            ).nlargest(max_add_per_round, "_confidence").drop(columns=["_confidence"])

        new_confident = new_confident.drop(columns=["mean_prob", "std_prob"])

        if auc < best_auc - 0.01:
            break
        if len(new_confident) == 0:
            break

        labeled_pool = pd.concat([labeled_pool, new_confident], ignore_index=True)
        unlabeled_pool = unlabeled_pool.drop(new_confident.index).drop(columns=["mean_prob", "std_prob"], errors="ignore")

    return baseline_auc, best_auc, best_round


# re-run with different split_seed to check the stability 
n_repeats = 15   # 15 times to determine the trend
baseline_aucs = []
best_aucs = []
best_rounds = []

for split_seed in range(n_repeats):
    baseline_auc, best_auc, best_round = run_iterative_ensemble(
        seed_set, unlabeled, num_cols, model, split_seed=split_seed
    )
    baseline_aucs.append(baseline_auc)
    best_aucs.append(best_auc)
    best_rounds.append(best_round)
    print(f"Seed {split_seed}: baseline AUC={baseline_auc:.3f} | best AUC={best_auc:.3f} at round {best_round}")

baseline_aucs = np.array(baseline_aucs)
best_aucs = np.array(best_aucs)

print("\n=== Stability summary ===")
print(f"Baseline AUC (round 0 only): {baseline_aucs.mean():.3f} +/- {baseline_aucs.std():.3f}")
print(f"Best AUC (after ensemble self-training): {best_aucs.mean():.3f} +/- {best_aucs.std():.3f}")
print(f"Splits where self-training improved: {(best_aucs > baseline_aucs).mean():.1%}")
print(f"Splits where self-training hurt (best < baseline): {(best_aucs < baseline_aucs).mean():.1%}")
print(f"Mean improvement: {(best_aucs - baseline_aucs).mean():.4f}")
print(f"Average stopping round: {np.mean(best_rounds):.1f}")

Seed 0: baseline AUC=1.000 | best AUC=1.000 at round 0
Seed 1: baseline AUC=1.000 | best AUC=1.000 at round 0
Seed 2: baseline AUC=0.920 | best AUC=0.940 at round 2
Seed 3: baseline AUC=0.920 | best AUC=0.960 at round 2
Seed 4: baseline AUC=0.940 | best AUC=0.980 at round 1
Seed 5: baseline AUC=0.880 | best AUC=0.920 at round 1
Seed 6: baseline AUC=0.960 | best AUC=0.980 at round 3
Seed 7: baseline AUC=0.940 | best AUC=0.980 at round 5
Seed 8: baseline AUC=0.980 | best AUC=0.980 at round 0
Seed 9: baseline AUC=0.820 | best AUC=0.860 at round 2
Seed 10: baseline AUC=1.000 | best AUC=1.000 at round 0
Seed 11: baseline AUC=0.960 | best AUC=0.960 at round 0
Seed 12: baseline AUC=0.880 | best AUC=0.900 at round 3
Seed 13: baseline AUC=0.980 | best AUC=0.980 at round 0
Seed 14: baseline AUC=0.840 | best AUC=0.840 at round 0

=== Stability summary ===
Baseline AUC (round 0 only): 0.935 +/- 0.056
Best AUC (after ensemble self-training): 0.952 +/- 0.049
Splits where self-training improved: 53.3

## FINAL MODEL - PREDICT ON THE FULL MERCHANT POPULATION

In [5]:
# rebuild the full merchant table from seed_set + unlabeled (together they cover all 4026 merchants)
pdf_merchant = pd.concat([seed_set, unlabeled], ignore_index=True)
print(f"pdf_merchant: {len(pdf_merchant)} rows")  # should be 4026

# threshold from 01_Label_merchants.ipynb, scaled to 0-1 to match merchant_fraud_score_final
threshold_merchant = (53.286933 + 61.923809) / 2
threshold_merchant_normalized = threshold_merchant / 100

# train final model on best_labeled_pool, then predict for all merchants
final_split_seed = 42
tr, te = train_test_split(
    seed_set, test_size=0.3, random_state=final_split_seed, stratify=seed_set["is_fraud"]
)

labeled_pool = tr.copy()
unlabeled_pool = unlabeled.copy()

best_auc = -1
best_labeled_pool = None
best_round = -1

for round_i in range(n_rounds):
    # train 5 bootstrap models like before
    ensemble_models = []
    for b in range(n_bootstrap):
        boot_sample = resample(labeled_pool, random_state=final_split_seed * 1000 + round_i * 100 + b)
        m_b = clone(model)
        m_b.fit(boot_sample[num_cols], boot_sample["is_fraud"])
        ensemble_models.append(m_b)

    # check AUC on test set 
    test_probs = np.array([m_b.predict_proba(te[num_cols])[:, 1] for m_b in ensemble_models])
    auc = roc_auc_score(te["is_fraud"], test_probs.mean(axis=0))

    # keep track of the best round
    if auc > best_auc:
        best_auc = auc
        best_labeled_pool = labeled_pool.copy()
        best_round = round_i

    if len(unlabeled_pool) == 0:
        break

    # predict on the remaining unlabeled merchants
    unlabeled_probs = np.array([m_b.predict_proba(unlabeled_pool[num_cols])[:, 1] for m_b in ensemble_models])
    mean_prob = unlabeled_probs.mean(axis=0)
    std_prob = unlabeled_probs.std(axis=0)
    unlabeled_pool = unlabeled_pool.copy()
    unlabeled_pool["mean_prob"] = mean_prob
    unlabeled_pool["std_prob"] = std_prob

    # only keep samples the models agree on and are confident about
    agree_mask = unlabeled_pool["std_prob"] <= max_std
    confident_fraud = unlabeled_pool[agree_mask & (unlabeled_pool["mean_prob"] > high_conf)].assign(is_fraud=1.0)
    confident_clean = unlabeled_pool[agree_mask & (unlabeled_pool["mean_prob"] < low_conf)].assign(is_fraud=0.0)
    new_confident = pd.concat([confident_fraud, confident_clean])

    # restriction on adding 30 new samples per round
    if len(new_confident) > max_add_per_round:
        new_confident = new_confident.assign(
            _confidence=np.abs(new_confident["mean_prob"] - 0.5)
        ).nlargest(max_add_per_round, "_confidence").drop(columns=["_confidence"])
    new_confident = new_confident.drop(columns=["mean_prob", "std_prob"])

    # stop if AUC gets worse or no samples are confident enough
    if auc < best_auc - 0.01 or len(new_confident) == 0:
        break

    # move new samples into labeled_pool, remove from unlabeled_pool
    labeled_pool = pd.concat([labeled_pool, new_confident], ignore_index=True)
    unlabeled_pool = unlabeled_pool.drop(new_confident.index).drop(columns=["mean_prob", "std_prob"], errors="ignore")

print(f"Final best AUC: {best_auc:.3f} at round {best_round}, labeled pool size: {len(best_labeled_pool)}")

# train the "official" final ensemble on best_labeled_pool 
final_ensemble = []
for b in range(n_bootstrap):
    boot_sample = resample(best_labeled_pool, random_state=9999 + b)
    m_b = clone(model)
    m_b.fit(boot_sample[num_cols], boot_sample["is_fraud"])
    final_ensemble.append(m_b)

# predict on merchant that don't have a real label
all_probs = np.array([m_b.predict_proba(unlabeled[num_cols])[:, 1] for m_b in final_ensemble])
unlabeled_final = unlabeled.copy()
unlabeled_final["pred_proba"] = all_probs.mean(axis=0)
print(f"Predicted for {len(unlabeled_final)} merchants without a ground-truth label")

pdf_merchant: 4026 rows
Final best AUC: 0.920 at round 2, labeled pool size: 93
Predicted for 3978 merchants without a ground-truth label


## CALIBRATED

In [6]:
from sklearn.calibration import CalibratedClassifierCV

# fit once on the full seed_set (48 merchants), no pseudo-labels
calibrated_model = CalibratedClassifierCV(clone(model), method="sigmoid", cv=5)
calibrated_model.fit(seed_set[num_cols], seed_set["is_fraud"])

# predict fraud probability for all unlabeled merchants
unlabeled_final = unlabeled.copy()
unlabeled_final["pred_proba"] = calibrated_model.predict_proba(unlabeled[num_cols])[:, 1]

# only trust merchants inside the p10-p90 range of seed_set
p10 = seed_set[num_cols].quantile(0.10)
p90 = seed_set[num_cols].quantile(0.90)
well_supported = np.logical_and.reduce(
    [(unlabeled[c] >= p10[c]) & (unlabeled[c] <= p90[c]) for c in num_cols]
)
print(f"Well-supported: {well_supported.sum()} / {len(unlabeled)}")

# outside that range, leave NaN instead of guessing
unlabeled_final.loc[~well_supported, "pred_proba"] = np.nan

# sanity check: seed_set fraud rate is 35.4%
print(f"Mean pred_proba (well-supported): {unlabeled_final['pred_proba'].mean():.3f}")

Well-supported: 1561 / 3978
Mean pred_proba (well-supported): 0.583


## MERGE AND SAVE

In [7]:
# merge predictions back into the main merchant table
pdf_merchant_final = pdf_merchant.merge(
    unlabeled_final[["merchant_abn", "pred_proba"]], on="merchant_abn", how="left"
)

# if merchant already has avg_merchant_fraud_prob, keep it (scaled to 0-1)
pdf_merchant_final["merchant_fraud_score_final"] = np.where(
    pdf_merchant_final["avg_merchant_fraud_prob"].notna(),
    pdf_merchant_final["avg_merchant_fraud_prob"] / 100,
    pdf_merchant_final["pred_proba"]
)

# apply threshold to get the final 0/1 label
pdf_merchant_final["is_fraud_merchant"] = np.where(
    pdf_merchant_final["merchant_fraud_score_final"].isna(), np.nan,
    (pdf_merchant_final["merchant_fraud_score_final"] > threshold_merchant_normalized).astype(float)
)

print(pdf_merchant_final["is_fraud_merchant"].value_counts(dropna=False))

# save under iter output to distinguish with pseudo-labels 
merchant_fraud_iter_output = pdf_merchant_final[["merchant_abn", "merchant_fraud_score_final", "is_fraud_merchant"]].copy()
merchant_fraud_iter_output.to_parquet("../../data/curated/merchant_fraud_labels_iter.parquet", index=False)
print(f"Saved: {len(merchant_fraud_iter_output)} merchants")

is_fraud_merchant
NaN    2417
1.0     833
0.0     776
Name: count, dtype: int64
Saved: 4026 merchants


In [8]:
# compare my labels with paulina's on merchants both of us labeled
bootstrap = pd.read_parquet("../../data/curated/merchant_fraud_labels_iter.parquet")
pseudo = pd.read_parquet("../../data/curated/merchant_fraud_labels.parquet")

both = bootstrap.merge(pseudo, on="merchant_abn", suffixes=("_bootstrap", "_pseudo"))
both = both.dropna(subset=["is_fraud_merchant_bootstrap", "is_fraud_merchant_pseudo"])

agree = (both["is_fraud_merchant_bootstrap"] == both["is_fraud_merchant_pseudo"]).mean()
print(f"Merchants labeled by both: {len(both)}")
print(f"Agreement: {agree:.1%}")
print(pd.crosstab(both["is_fraud_merchant_bootstrap"], both["is_fraud_merchant_pseudo"]))

Merchants labeled by both: 1609
Agreement: 100.0%
is_fraud_merchant_pseudo     0.0  1.0
is_fraud_merchant_bootstrap          
0.0                          776    0
1.0                            0  833
